<a href="https://colab.research.google.com/github/Aradhyagodambe/quire/blob/main/quire.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [1]:
!pip install -qU llama-index-postprocessor-sbert-rerank pymupdf4llm llama-index transformers accelerate bitsandbytes sentence-transformers llama-index-embeddings-huggingface llama-index-llms-huggingface

     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 44.0/44.0 kB 1.6 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 176.4/176.4 kB 10.7 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 25.8/25.8 MB 40.8 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 42.9/42.9 MB 14.7 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 11.9/11.9 MB 39.7 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 41.0/41.0 MB 20.1 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 12.0/12.0 MB 52.1 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 566.4/566.4 kB 19.9 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 1.8/1.8 MB 27.7 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 166.8/166.8 kB 9.8 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 1.1/1.1 MB 59.6 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 23.1/23.1 MB 25.6 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 51.0/5

In [2]:
import torch
from transformers import BitsAndBytesConfig
from llama_index.core import Settings
from llama_index.llms.huggingface import HuggingFaceLLM
from llama_index.embeddings.huggingface import HuggingFaceEmbedding

In [3]:
bnb_config = BitsAndBytesConfig(
    load_in_4bit= True,
    bnb_4bit_compute_dtype= torch.float16,
    bnb_4bit_quant_type= "nf4",
    bnb_4bit_use_double_quant = True
)

In [4]:
from transformers import AutoTokenizer

zephyr_tokenizer = AutoTokenizer.from_pretrained("HuggingFaceH4/zephyr-7b-beta")

def messages_to_prompt(messages):
    chat = [{"role": m.role.value, "content": m.content} for m in messages]
    return zephyr_tokenizer.apply_chat_template(chat, tokenize=False, add_generation_prompt=True)

def completion_to_prompt(completion):
    return zephyr_tokenizer.apply_chat_template(
        [{"role": "user", "content": completion}], tokenize=False, add_generation_prompt=True
    )

/usr/local/lib/python3.13/dist-packages/huggingface_hub/utils/_auth.py:94: UserWarning: 
The secret `HF_TOKEN` does not exist in your Colab secrets.
To authenticate with the Hugging Face Hub, create a token in your settings tab (https://huggingface.co/settings/tokens), set it as secret in your Google Colab and restart your session.
You will be able to reuse this secret in all of your notebooks.
Please note that authentication is recommended but still optional to access public models or datasets.
  warnings.warn(


tokenizer_config.json: 0.00B [00:00, ?B/s]

tokenizer.model:   0%|          | 0.00/493k [00:00<?, ?B/s]

tokenizer.json: 0.00B [00:00, ?B/s]

added_tokens.json:   0%|          | 0.00/42.0 [00:00<?, ?B/s]

special_tokens_map.json:   0%|          | 0.00/168 [00:00<?, ?B/s]

In [5]:
llm = HuggingFaceLLM(
    model_name="HuggingFaceH4/zephyr-7b-beta",
    tokenizer_name="HuggingFaceH4/zephyr-7b-beta",
    context_window=8192,
    max_new_tokens=512,
    model_kwargs={"quantization_config": bnb_config},
    generate_kwargs={"temperature": 0.1, "do_sample": True},
    messages_to_prompt=messages_to_prompt,
    completion_to_prompt=completion_to_prompt,
    device_map="auto",
)

config.json:   0%|          | 0.00/638 [00:00<?, ?B/s]

model.safetensors.index.json: 0.00B [00:00, ?B/s]

Fetching 8 files:   0%|          | 0/8 [00:01<?, ?it/s]

model-00002-of-00008.safetensors:   0%|          | 0.00/1.95G [00:00<?, ?B/s]

model-00004-of-00008.safetensors:   0%|          | 0.00/1.95G [00:00<?, ?B/s]

...

model-00007-of-00008.safetensors:   0%|          | 0.00/1.98G [00:00<?, ?B/s]

model-00006-of-00008.safetensors:   0%|          | 0.00/1.95G [00:00<?, ?B/s]

model-00008-of-00008.safetensors:   0%|          | 0.00/816M [00:00<?, ?B/s]

Loading checkpoint shards:   0%|          | 0/8 [00:00<?, ?it/s]

generation_config.json:   0%|          | 0.00/111 [00:00<?, ?B/s]

In [6]:
Settings.llm = llm
Settings.embed_model = HuggingFaceEmbedding(model_name = "sentence-transformers/all-MiniLM-L6-v2")
Settings.chunk_size = 512
Settings.chunk_overlap = 50

modules.json:   0%|          | 0.00/349 [00:00<?, ?B/s]

config_sentence_transformers.json:   0%|          | 0.00/116 [00:00<?, ?B/s]

README.md: 0.00B [00:00, ?B/s]

sentence_bert_config.json:   0%|          | 0.00/53.0 [00:00<?, ?B/s]

config.json:   0%|          | 0.00/612 [00:00<?, ?B/s]

...

tokenizer_config.json:   0%|          | 0.00/350 [00:00<?, ?B/s]

vocab.txt: 0.00B [00:00, ?B/s]

tokenizer.json: 0.00B [00:00, ?B/s]

special_tokens_map.json:   0%|          | 0.00/112 [00:00<?, ?B/s]

config.json:   0%|          | 0.00/190 [00:00<?, ?B/s]

In [7]:
import pymupdf4llm
from llama_index.core import Document, VectorStoreIndex
from google.colab import files

In [8]:
uploaded = files.upload()

documents = []

Saving research ppr 6.pdf to research ppr 6.pdf
Saving research ppr 5.pdf to research ppr 5.pdf
Saving research ppr 4.pdf to research ppr 4.pdf
Saving research ppr 3.pdf to research ppr 3.pdf
Saving research ppr 2.pdf to research ppr 2.pdf
Saving research ppr.pdf to research ppr.pdf


In [9]:
import re

def clean_filename(pdf_path):

    return re.sub(r" \(\d+\)(?=\.\w+$)", "", pdf_path)

In [10]:
for pdf_path in uploaded.keys():
  print(pdf_path)

  page_data = pymupdf4llm.to_markdown(pdf_path, page_chunks = True)

  for page in page_data:
    current_page_num = page.get("metadata", {}).get("page",0) +1

    doc = Document(
        text = page["text"],
        metadata = {
            "filename" : pdf_path,
            "page_number" : current_page_num
        }
    )
    documents.append(doc)

research ppr 6.pdf

=== Document parser messages ===
Using Tesseract for OCR processing.
OCR on page.number=2/3.
OCR on page.number=3/4.
OCR on page.number=6/7.
OCR on page.number=8/9.
OCR on page.number=9/10.
OCR on page.number=11/12.
OCR on page.number=13/14.
research ppr 5.pdf

=== Document parser messages ===
Using Tesseract for OCR processing.
OCR on page.number=3/4.
OCR on page.number=5/6.
OCR on page.number=7/8.
OCR on page.number=8/9.
OCR on page.number=9/10.
OCR on page.number=10/11.
OCR on page.number=13/14.
research ppr 4.pdf

=== Document parser messages ===
Using Tesseract for OCR processing.
OCR on page.number=4/5.
OCR on page.number=14/15.
OCR on page.number=18/19.
OCR on page.number=19/20.
 page.number=9/10.
OCR on page.number=10/11.
research ppr 3.pdf

=== Document parser messages ===
Using Tesseract for OCR processing.
OCR on page.number=2/3.
OCR on page.number=3/4.
OCR on page.number=6/7.
OCR on page.number=8/9.
OCR on page.number=9/10.
OCR on page.number=11/12.
OCR 

In [11]:
print(f"Building Vector Index {len(documents)} ")
index = VectorStoreIndex.from_documents(documents)

Building Vector Index 103 


In [12]:
from llama_index.core.memory import ChatMemoryBuffer
from llama_index.core.vector_stores import MetadataFilter, MetadataFilters, FilterOperator
from llama_index.postprocessor.sbert_rerank import SentenceTransformerRerank

In [13]:
reranker = SentenceTransformerRerank(
    model = "cross-encoder/ms-marco-MiniLM-L-6-v2",
    top_n = 10
)

shared_memory = ChatMemoryBuffer.from_defaults(token_limit = 1500)

config.json:   0%|          | 0.00/794 [00:00<?, ?B/s]

model.safetensors:   0%|          | 0.00/90.9M [00:00<?, ?B/s]

tokenizer_config.json: 0.00B [00:00, ?B/s]

vocab.txt: 0.00B [00:00, ?B/s]

tokenizer.json: 0.00B [00:00, ?B/s]

special_tokens_map.json:   0%|          | 0.00/132 [00:00<?, ?B/s]

In [14]:
from llama_index.core import PromptTemplate

# 1. Create a simpler, highly direct prompt for the 7B model
custom_condense_prompt = PromptTemplate(
    "Given the following conversation history and a follow-up question, rewrite the follow-up question into a standalone query that contains all necessary context.\n\n"
    "Chat History:\n"
    "{chat_history}\n\n"
    "Follow Up Input: {question}\n\n"
    "Standalone query:"
)

In [15]:
def get_chat_engine(target_files=None):

    filters = None

    if target_files:

        if isinstance(target_files, str):
            target_files = [target_files]

        filters = MetadataFilters(
            filters=[
                MetadataFilter(
                    key="filename",
                    value=target_files,
                    operator=FilterOperator.IN
                )
            ]
        )

    return index.as_chat_engine(
        chat_mode="condense_plus_context",
        memory=shared_memory,
        filters=filters,
        similarity_top_k=24,
        node_postprocessors=[reranker],
        condense_prompt = custom_condense_prompt
    )

In [16]:
global_engine = get_chat_engine(target_files = None)

query_1 = "Summarize the abstracts and conclusions found in these documents."
print(f"\nQuestion :  {query_1}")
response_1 = global_engine.chat(query_1)
print(f"\nAnswer:\n {response_1.response}\n")


targeted_engine = get_chat_engine(target_files="research ppr 2.pdf")

query_2 = "How does this specific paper address those themes?"
print(f"\nQuestion : {query_2}")
response_2 = targeted_engine.chat(query_2)
print(f"\nAnswer:\n {response_2.response}\n")

Setting `pad_token_id` to `eos_token_id`:2 for open-end generation.



Question :  Summarize the abstracts and conclusions found in these documents.


Setting `pad_token_id` to `eos_token_id`:2 for open-end generation.



Answer:
 Document 1: "Research PPR 4"

Abstract: The abstract briefly introduces the proposed model for anomaly detection in financial transactions using machine learning techniques. It highlights the use of a confusion matrix to evaluate the model's effectiveness and the importance of reducing overfitting and validation in the context of fraud detection.

Conclusion: The conclusion summarizes the simulation results, which indicate that the proposed model accurately identifies positive and negative classes, with high true positive and true negative rates. The heatmap of the confusion matrix further supports the effectiveness of the model.

Document 2: "Research PPR 2"

Abstract: The abstract explains the features used in the dataset, which include step, type, amount, identification credentials, account balances, and flags for fraud and large transactions. It also mentions the high imbalance in the dataset and the focus on transfer and cash-out transactions.

Conclusion: The conclusion

Setting `pad_token_id` to `eos_token_id`:2 for open-end generation.



Answer:
 The paper "Research PPR 2" addresses the themes of machine learning techniques for fraud detection and data imbalance in the following ways:

1. Dataset: The paper uses a dataset of financial transactions, which includes features such as step, type, amount, identification credentials, account balances, and flags for fraud and large transactions. The dataset is highly imbalanced, with only 0.001% of the transactions being fraudulent.

2. Machine Learning Algorithms: The paper uses five machine learning algorithms, including Bernoulli naïve bayes, multinomial naïve bayes, passive aggressive classifier, stochastic gradient descent, and perceptron, to select the best model for deployment in the live environment. The authors note that the Bayesian algorithm is probabilistic and considers the presumption that a feature under a class does not depend on the other features of that class, considering each feature to be class independent.

3. Experimentation: The paper conducts thorough

In [21]:
print("\n--- Source Citations for Global Search---")
for i, node in enumerate(response_1.source_nodes, 1):
    filename = node.node.metadata.get("filename", "Unknown")
    page_num = node.node.metadata.get("page_number", "Unknown")
    score = round(node.score, 3) if node.score is not None else "N/A"
    print(f"Source {i}: {filename} (Page {page_num}) - Relevance Score: {score}")


--- Source Citations for Global Search---
Source 1: research ppr 4.pdf (Page 1) - Relevance Score: -6.491
Source 2: research ppr 2.pdf (Page 1) - Relevance Score: -8.347
Source 3: research ppr 3.pdf (Page 1) - Relevance Score: -9.812


In [20]:
print("\n--- Source Citations for Targeted Search---")
for i, node in enumerate(response_2.source_nodes, 1):
    filename = node.node.metadata.get("filename", "Unknown")
    page_num = node.node.metadata.get("page_number", "Unknown")
    score = round(node.score, 3) if node.score is not None else "N/A"
    print(f"Source {i}: {filename} (Page {page_num}) - Relevance Score: {score}")


--- Source Citations for Targeted Search---
Source 1: research ppr 2.pdf (Page 1) - Relevance Score: -0.55
Source 2: research ppr 2.pdf (Page 1) - Relevance Score: -0.673
Source 3: research ppr 2.pdf (Page 1) - Relevance Score: -0.689


In [ ]:
# import gc
# del llm
# gc.collect()
# torch.cuda.empty_cache()